In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 320
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
model.print_trainable_parameters()

Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [ ]:
from google.colab import files
uploaded = files.upload()  # select v2_final_train.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="v2_final_train.jsonl", split="train")
print(dataset)
print(dataset[0])

Saving v2_final_train.jsonl to v2_final_train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 232
})
{'system': 'You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale.', 'user': 'MS: price $530.49, RSI 50, trading resumed after a halt with no explanation published, price roughly unchanged.', 'assistant': '{"action": "hold", "confidence": 0.34, "rationale": "The available information is incomplete or unreliable, which makes any directional call speculative. Waiting for clean, confirmed data is more prudent than acting on an ambiguous signal."}'}


In [ ]:
def format_example(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)
print(dataset[0]["text"])

lengths = [len(tokenizer(ex["text"])["input_ids"]) for ex in dataset]
print("Token lengths — min:", min(lengths), "max:", max(lengths), "avg:", sum(lengths)/len(lengths))

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

<|im_start|>system
You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale.<|im_end|>
<|im_start|>user
MS: price $530.49, RSI 50, trading resumed after a halt with no explanation published, price roughly unchanged.<|im_end|>
<|im_start|>assistant
{"action": "hold", "confidence": 0.34, "rationale": "The available information is incomplete or unreliable, which makes any directional call speculative. Waiting for clean, confirmed data is more prudent than acting on an ambiguous signal."}<|im_end|>

Token lengths — min: 109 max: 160 avg: 135.8706896551724


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/232 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/232 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 232 | Num Epochs = 2 | Total steps = 58
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.490118
2,2.265883
3,2.355052
4,2.024843
5,2.011225
6,1.969023
7,1.853822
8,1.859214
9,1.661900
10,1.483508


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-58/tokenizer_config.json.


In [ ]:
model.save_pretrained("qwen_trading_lora_v2")
tokenizer.save_pretrained("qwen_trading_lora_v2")

!zip -r qwen_trading_lora_v2.zip qwen_trading_lora_v2
files.download("qwen_trading_lora_v2.zip")

Unsloth: Restored added_tokens_decoder metadata in qwen_trading_lora_v2/tokenizer_config.json.


  adding: qwen_trading_lora_v2/ (stored 0%)
  adding: qwen_trading_lora_v2/adapter_model.safetensors (deflated 8%)
  adding: qwen_trading_lora_v2/tokenizer_config.json (deflated 89%)
  adding: qwen_trading_lora_v2/adapter_config.json (deflated 59%)
  adding: qwen_trading_lora_v2/chat_template.jinja (deflated 71%)
  adding: qwen_trading_lora_v2/README.md (deflated 65%)
  adding: qwen_trading_lora_v2/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

FastLanguageModel.for_inference(model)
test_prompt = [
    {"role": "system", "content": "You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale."},
    {"role": "user", "content": "COIN: price $210.40, RSI 74, 5d momentum +11.8%, price up sharply after Bitcoin rallied 9% this week, volume 2x average."},
]
inputs = tokenizer.apply_chat_template(test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=False)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"action": "sell", "confidence": 0.63, "rationale": "RSI of 74 combined with a 11.8% five-day move driven by an external rally rather than company-specific fundamentals suggests the move is borrowed momentum. High odds of reversion once the external driver fades."}


In [ ]:
import json, re

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

def run_eval(m, tok, eval_data, label=""):
    FastLanguageModel.for_inference(m)
    results = []
    for ex in eval_data:
        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
        ]
        inputs = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        outputs = m.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=False)
        raw = tok.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        parsed = extract_json(raw)
        true_label = json.loads(ex["assistant"])
        results.append({
            "user": ex["user"], "raw_output": raw, "parsed": parsed,
            "valid_json": parsed is not None,
            "true_action": true_label["action"],
            "pred_action": parsed["action"] if parsed else None,
            "correct": parsed is not None and parsed.get("action") == true_label["action"],
            "pred_confidence": parsed.get("confidence") if parsed else None,
        })
    return results

def summarize(results, label=""):
    n = len(results)
    valid = [r for r in results if r["valid_json"]]
    n_valid = len(valid)
    correct = [r for r in valid if r["correct"]]
    accuracy = len(correct) / n_valid if n_valid else 0
    print(f"=== {label} ===")
    print(f"Total examples: {n}")
    print(f"Valid JSON: {n_valid}/{n} ({100*n_valid/n:.1f}%)")
    print(f"Accuracy (on valid outputs): {len(correct)}/{n_valid} ({100*accuracy:.1f}%)")
    print("\nPer-class breakdown:")
    for action in ["buy", "sell", "hold"]:
        class_examples = [r for r in valid if r["true_action"] == action]
        class_correct = [r for r in class_examples if r["correct"]]
        n_class = len(class_examples)
        acc = len(class_correct)/n_class if n_class else 0
        pred_dist = {}
        for r in class_examples:
            pred_dist[r["pred_action"]] = pred_dist.get(r["pred_action"], 0) + 1
        print(f"  {action:5s}: {len(class_correct)}/{n_class} correct ({100*acc:.1f}%)  |  predicted as: {pred_dist}")
    print("\nConfidence calibration:")
    buckets = {"low (0-0.5)": [], "mid (0.5-0.7)": [], "high (0.7-1.0)": []}
    for r in valid:
        c = r["pred_confidence"] or 0
        if c < 0.5: buckets["low (0-0.5)"].append(r)
        elif c < 0.7: buckets["mid (0.5-0.7)"].append(r)
        else: buckets["high (0.7-1.0)"].append(r)
    for name, bucket in buckets.items():
        if bucket:
            acc = sum(r["correct"] for r in bucket)/len(bucket)
            print(f"  {name}: n={len(bucket)}, accuracy={100*acc:.1f}%")
    return {"n": n, "valid_pct": 100*n_valid/n, "accuracy": 100*accuracy}

In [ ]:
import os
print("v2_final_eval.jsonl exists:", os.path.exists("v2_final_eval.jsonl"))
print("ood_eval.jsonl exists:", os.path.exists("ood_eval.jsonl"))

v2_final_eval.jsonl exists: True
ood_eval.jsonl exists: False


In [ ]:
uploaded = files.upload()  # select BOTH v2_final_eval.jsonl and ood_eval.jsonl in this one dialog

Saving ood_eval.jsonl to ood_eval.jsonl


In [ ]:
uploaded = files.upload()  # select v2_final_eval.jsonl AND ood_eval.jsonl together

eval_data = load_jsonl("v2_final_eval.jsonl")
ood_data = load_jsonl("ood_eval.jsonl")

ft_v2_templated = run_eval(model, tokenizer, eval_data, label="v2-templated")
summarize(ft_v2_templated, label="Fine-tuned v2 — templated eval")

ft_v2_ood = run_eval(model, tokenizer, ood_data, label="v2-ood")
summarize(ft_v2_ood, label="Fine-tuned v2 — OOD eval")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saving v2_final_eval.jsonl to v2_final_eval (1).jsonl


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Fine-tuned v2 — templated eval ===
Total examples: 80
Valid JSON: 80/80 (100.0%)
Accuracy (on valid outputs): 72/80 (90.0%)

Per-class breakdown:
  buy  : 27/29 correct (93.1%)  |  predicted as: {'buy': 27, 'hold': 1, 'sell': 1}
  sell : 26/28 correct (92.9%)  |  predicted as: {'sell': 26, 'hold': 2}
  hold : 19/23 correct (82.6%)  |  predicted as: {'hold': 19, 'sell': 3, 'buy': 1}

Confidence calibration:
  low (0-0.5): n=24, accuracy=87.5%
  mid (0.5-0.7): n=29, accuracy=86.2%
  high (0.7-1.0): n=27, accuracy=96.3%


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Fine-tuned v2 — OOD eval ===
Total examples: 10
Valid JSON: 10/10 (100.0%)
Accuracy (on valid outputs): 7/10 (70.0%)

Per-class breakdown:
  buy  : 2/3 correct (66.7%)  |  predicted as: {'sell': 1, 'buy': 2}
  sell : 2/2 correct (100.0%)  |  predicted as: {'sell': 2}
  hold : 3/5 correct (60.0%)  |  predicted as: {'hold': 3, 'sell': 2}

Confidence calibration:
  low (0-0.5): n=3, accuracy=100.0%
  mid (0.5-0.7): n=5, accuracy=40.0%
  high (0.7-1.0): n=2, accuracy=100.0%


{'n': 10, 'valid_pct': 100.0, 'accuracy': 70.0}